In [ ]:
import matplotlib
matplotlib.use("Agg")  # Use non-interactive backend for plot rendering
import yaml
import pandas as pd
from pathlib import Path
from IPython.display import Image, display
from risk_validation.core.metrics.impl import pd as pd_metrics  # noqa: F401
from risk_validation.core.services.pd.metrics_service import PDMetricsService
from risk_validation.core.utils.plots import plot_roc, plot_gini, plot_ks_cdf_with_maxgap
from risk_validation.core.utils.rag import rag_for_metric

# Define data directories and load data
DATA_DIR = Path('..') / 'data'
csv_path = DATA_DIR / 'sample.csv'
df = pd.read_csv(csv_path)

# Setup column mappings based on provided mappings
params = {
    'y_col': 'default_flag',
    'p_col': 'pred_br',
    'score_col': 'score_pd',
    'band_col': 'BAND'
}

dev_year = 2019
val_year = 2020

df_dev = df[df['score_year'] == dev_year]
df_val = df[df['score_year'] == val_year]

# Compute metrics
service = PDMetricsService(['auc_roc', 'gini', 'ks'])
dev_result = service.compute(df_dev, params)
val_result = service.compute(df_val, params)

# Load RAG thresholds
rag_path = Path('../configs/validation_rag.yaml')
with open(rag_path) as f:
    rag_config = yaml.safe_load(f)

# Calculate RAG and display metrics and plots
plot_dir = Path('plots')
plot_dir.mkdir(exist_ok=True)

for metric_id in ['auc_roc', 'gini', 'ks']:
    dev_row = dev_result[dev_result['metric_id'] == metric_id]
    val_row = val_result[val_result['metric_id'] == metric_id]
    
    if not dev_row.empty and not val_row.empty:
        dev_value = dev_row['value'].iloc[0]
        val_value = val_row['value'].iloc[0]

        # Calculate RAG
        rag_result = rag_for_metric(
            metric_id=metric_id,
            cfg=rag_config['pd'],
            dev_value=dev_value,
            val_value=val_value
        )

        print(f"{metric_id.upper()}: DEV={dev_value:.4f}, VAL={val_value:.4f}, RAG: {rag_result['status']}")

        if metric_id == 'auc_roc':
            plot_path = plot_roc(
                y_true=df_val['default_flag'],
                y_score=df_val['score_pd'],
                title='ROC Curve',
                out_path=plot_dir / 'roc.png'
            )
        elif metric_id == 'gini':
            plot_path = plot_gini(
                y_true=df_val['default_flag'],
                y_score=df_val['score_pd'],
                title='Gini Plot',
                out_path=plot_dir / 'gini.png'
            )
        elif metric_id == 'ks':
            plot_path, _ = plot_ks_cdf_with_maxgap(
                y_true=df_val['default_flag'],
                y_score=df_val['score_pd'],
                title='KS CDF',
                out_path=plot_dir / 'ks.png'
            )
        
        display(Image(filename=str(plot_path)))